# 🔍 Day 04: Binary Search — The O(log n) Superpower

---

## More Than Just "Find a Number in a Sorted Array"

Most people think binary search is just finding a target in a sorted array. That's like saying a Swiss army knife is just a blade — you're missing 90% of what it can do.

Binary search is really about this: **whenever you can split the problem in half and throw away one half, you've got an O(log n) solution.**

**Keywords that scream binary search:**
- "Sorted array" + "find"
- "Minimum/maximum that satisfies a condition"
- "Search in rotated array"
- "Koko eating bananas" (yes, seriously)

---

## 🧠 The Classic Template

There are a few binary search templates floating around. Here's the one that handles the most cases cleanly:

```
The KEY question at each step:

Is the answer in the LEFT half or the RIGHT half?
→ Check the MIDDLE element to decide
→ Throw away the half that can't contain the answer
→ Repeat until converged
```

In [ ]:
# === Template 1: Standard Binary Search ===
def binary_search(nums, target):
    """
    Find target in sorted array. Returns index or -1.
    Time: O(log n), Space: O(1)
    """
    left, right = 0, len(nums) - 1
    
    while left <= right:  # Note: <= not <
        mid = left + (right - left) // 2  # Avoids overflow (matters in other languages)
        
        if nums[mid] == target:
            return mid
        elif nums[mid] < target:
            left = mid + 1
        else:
            right = mid - 1
    
    return -1

nums = [1, 3, 5, 7, 9, 11, 13]
print("Find 7:", binary_search(nums, 7))   # 3
print("Find 6:", binary_search(nums, 6))   # -1

In [ ]:
# === Template 2: Find leftmost / rightmost position ===
# This is the one you need for "find first/last occurrence" type problems

import bisect

def find_left(nums, target):
    """Find first position where target could be inserted (leftmost target)."""
    left, right = 0, len(nums)
    while left < right:  # Note: < not <=
        mid = (left + right) // 2
        if nums[mid] < target:
            left = mid + 1
        else:
            right = mid
    return left

def find_right(nums, target):
    """Find first position AFTER target (one past rightmost target)."""
    left, right = 0, len(nums)
    while left < right:
        mid = (left + right) // 2
        if nums[mid] <= target:  # Note: <= instead of <
            left = mid + 1
        else:
            right = mid
    return left

nums = [1, 2, 2, 2, 3, 4]
print(f"Left position of 2: {find_left(nums, 2)}")   # 1
print(f"Right position of 2: {find_right(nums, 2) - 1}")  # 3
print(f"Count of 2s: {find_right(nums, 2) - find_left(nums, 2)}")  # 3

# Python's built-in does the same thing:
print(f"bisect_left: {bisect.bisect_left(nums, 2)}")   # 1
print(f"bisect_right: {bisect.bisect_right(nums, 2)}")  # 4

---

## 🎯 Problem 1: Search in Rotated Sorted Array (LC #33)

This is the **classic hard binary search** problem. Shows up constantly.

```
Array was sorted, then rotated at some pivot:
[4, 5, 6, 7, 0, 1, 2]  ← was [0,1,2,4,5,6,7] rotated at index 3

Find target in O(log n).
```

**The trick:** At any midpoint, one half is always sorted. Figure out which half is sorted, then check if target is in that sorted half.

In [ ]:
def search_rotated(nums, target):
    """
    Search in Rotated Sorted Array
    Time: O(log n), Space: O(1)
    """
    left, right = 0, len(nums) - 1
    
    while left <= right:
        mid = (left + right) // 2
        
        if nums[mid] == target:
            return mid
        
        # Which half is sorted?
        if nums[left] <= nums[mid]:  # Left half is sorted
            if nums[left] <= target < nums[mid]:  # Target in left sorted half?
                right = mid - 1
            else:
                left = mid + 1
        else:  # Right half is sorted
            if nums[mid] < target <= nums[right]:  # Target in right sorted half?
                left = mid + 1
            else:
                right = mid - 1
    
    return -1

print(search_rotated([4, 5, 6, 7, 0, 1, 2], 0))  # 4
print(search_rotated([4, 5, 6, 7, 0, 1, 2], 3))  # -1
print(search_rotated([1], 0))                       # -1

---

## 🎯 Problem 2: Find Minimum in Rotated Sorted Array (LC #153)

```
Find the minimum element in a rotated sorted array.
Input: [3, 4, 5, 1, 2]  →  Output: 1
```

**Key insight:** The minimum is at the "rotation point" — where the array breaks from sorted order.

In [ ]:
def findMin(nums):
    """
    Find Minimum in Rotated Sorted Array
    Time: O(log n), Space: O(1)
    """
    left, right = 0, len(nums) - 1
    
    while left < right:
        mid = (left + right) // 2
        
        if nums[mid] > nums[right]:  # Min is in right half
            left = mid + 1
        else:  # Min is in left half (including mid)
            right = mid
    
    return nums[left]

print(findMin([3, 4, 5, 1, 2]))   # 1
print(findMin([4, 5, 6, 7, 0, 1, 2]))  # 0
print(findMin([1]))                # 1

---

## 🎯 Problem 3: Koko Eating Bananas (LC #875)

This is **binary search on the answer** — the advanced pattern that's becoming really common.

```
Koko has piles of bananas. She can eat k bananas/hour.
Guards return in h hours. What's the minimum k so she finishes?

piles = [3, 6, 7, 11], h = 8  →  Output: 4
```

**The insight:** We're not searching through an array — we're searching through **possible values of k** (1 to max(piles)). For each k, we can check if Koko finishes in time. This check is monotonic: if k works, k+1 also works. So binary search!

In [ ]:
import math

def minEatingSpeed(piles, h):
    """
    Koko Eating Bananas — Binary Search on Answer
    Time: O(n * log(max_pile)), Space: O(1)
    """
    def can_finish(speed):
        """Can Koko finish all piles at this speed within h hours?"""
        hours_needed = sum(math.ceil(pile / speed) for pile in piles)
        return hours_needed <= h
    
    # Binary search between speed 1 and max(piles)
    left, right = 1, max(piles)
    
    while left < right:
        mid = (left + right) // 2
        
        if can_finish(mid):  # This speed works, but maybe a slower one does too
            right = mid
        else:               # Too slow, need to eat faster
            left = mid + 1
    
    return left

print(minEatingSpeed([3, 6, 7, 11], 8))    # 4
print(minEatingSpeed([30, 11, 23, 4, 20], 5))  # 30
print(minEatingSpeed([30, 11, 23, 4, 20], 6))  # 23

**Binary search on the answer** is a game-changer. The pattern:
1. Define a search range for the answer
2. Write a `check(mid)` function that returns True/False
3. Binary search to find the boundary

---

## 🗺️ Binary Search Pattern Map

```
╔═══════════════════════════════════════════════════════════════════════╗
║  BINARY SEARCH VARIATIONS                                            ║
╠═══════════════════════════════════════════════════════════════════════╣
║                                                                       ║
║  1. STANDARD: Find exact target in sorted array                      ║
║     while left <= right, check ==, shrink accordingly                ║
║                                                                       ║
║  2. BOUNDARY: Find first/last occurrence, insertion point            ║
║     while left < right, bisect_left / bisect_right                   ║
║                                                                       ║
║  3. ROTATED ARRAY: One half is always sorted                         ║
║     Check which half is sorted, decide based on that                 ║
║                                                                       ║
║  4. SEARCH ON ANSWER: Binary search the result space                 ║
║     Write check(mid), binary search on feasibility boundary          ║
║     → Koko Bananas, Split Array Largest Sum, Ship Packages           ║
║                                                                       ║
╚═══════════════════════════════════════════════════════════════════════╝
```

---

## 📝 Practice Problems

| # | Problem | Difficulty | Variation |
|---|---------|------------|----------|
| 704 | Binary Search | Easy | Standard |
| 35 | Search Insert Position | Easy | Boundary |
| 34 | Find First and Last Position | Medium | Boundary |
| 33 | Search in Rotated Array | Medium | Rotated |
| 153 | Find Min in Rotated Array | Medium | Rotated |
| 875 | Koko Eating Bananas | Medium | Search on answer |
| 1011 | Capacity to Ship Packages | Medium | Search on answer |
| 4 | Median of Two Sorted Arrays | Hard | Advanced |

---

## ✅ Key Takeaways

```
┌─────────────────────────────────────────────────────────────────────┐
│  📌 Binary search = eliminate half the search space each step      │
├─────────────────────────────────────────────────────────────────────┤
│  📌 left <= right for exact match, left < right for boundary      │
├─────────────────────────────────────────────────────────────────────┤
│  📌 Rotated array: one half is ALWAYS sorted — use that!          │
├─────────────────────────────────────────────────────────────────────┤
│  📌 "Search on answer" is binary search on the result space       │
├─────────────────────────────────────────────────────────────────────┤
│  📌 Use bisect module for quick left/right boundary searches      │
└─────────────────────────────────────────────────────────────────────┘
```

---

## ➡️ Next Up: Kadane's Algorithm & Subarray Patterns

The algorithm that solves Maximum Subarray in one pass.

---

### 🎉 Binary Search Mastered!
From sorted arrays to eating bananas — you now see binary search everywhere.